In [ ]:
!pip uninstall -y numpy faiss-cpu scikit-learn scipy sentence-transformers transformers
!rm -rf ~/.cache/pip

!pip install --no-cache-dir numpy==1.26.4
!pip install --no-cache-dir faiss-cpu scikit-learn scipy
!pip install --no-cache-dir transformers accelerate bitsandbytes sentencepiece
!pip install --no-cache-dir sentence-transformers tqdm joblib
!pip install --no-cache-dir scispacy
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz

print("✓ All dependencies installed")

import os
 # forces runtime restart — this is intentional

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.17.1
Uninstalling scipy-1.17.1:
  Successfully uninstalled scipy-1.17.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 224.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xgboost 3.2.0 requires scipy, which is not installed.
plotnine 0.14.5 requires scipy>=1.8.0, which is not installed.
stumpy 1.13.0 requires scipy>=1.10, which is not installed.
cuml-cu12 26.2.0 requires scikit-learn>=1.5, which is not installed.
cuml-cu12 26.2.0 requires scipy>=1.13.0, which is not installed.
pymc 5.28.4 requires scipy>=1.4.1, which is not installed.
yellowbrick 1.5 requires scikit-learn>=1.0.0, which is not installed.
yellowb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 352.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 366.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 309.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
 

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✓ Drive mounted')

Mounted at /content/drive
✓ Drive mounted


In [ ]:
# ── CONFIG — adjust paths if needed ──────────────────────────────────────────
TXT_PATH = '/content/drive/MyDrive/merged_final3.txt'   # <- your file on Drive
DATA_DIR = '/content/drive/MyDrive/first_aid_db_4'     # <- output folder on Drive

import os
os.makedirs(DATA_DIR, exist_ok=True)
print(f'✓ Output dir ready: {DATA_DIR}')

✓ Output dir ready: /content/drive/MyDrive/first_aid_db_4


In [ ]:
!pip install rank-bm25
import os, re, json, pickle
import numpy as np
from tqdm.notebook import tqdm

import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

import spacy
import scispacy
from scispacy.linking import EntityLinker

print('✓ Imports done')

✓ Imports done


In [ ]:
# ── MODEL CONFIG ──────────────────────────────────────────────────────────────
EMBED_MODEL         = 'sentence-transformers/all-mpnet-base-v2'
CHUNK_WORD_LIMIT    = 140
MIN_CHUNK_WORDS     = 4
SIM_THRESHOLD       = 0.55
CUI_SCORE_THRESHOLD = 0.4
OVERLAP_SENTENCES   = 1
print('✓ Config ready')

✓ Config ready


In [ ]:
# ── Load models ───────────────────────────────────────────────────────────────
print('Loading scispacy...')
nlp = spacy.load('en_core_sci_md')

if 'scispacy_linker' not in nlp.pipe_names:
    nlp.add_pipe(
        'scispacy_linker',
        config={
            'resolve_abbreviations': True,
            'linker_name': 'umls'
        }
    )

print('Loading embedding model...')
embed_model = SentenceTransformer(EMBED_MODEL)

print('✓ Models loaded')

Loading scispacy...


/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/tfidf_vectors_sparse.npz not found in cache, downloading to /tmp/tmpodzi748b


100%|██████████| 492M/492M [00:08<00:00, 64.0MiB/s]


Finished download, copying /tmp/tmpodzi748b to cache at /root/.scispacy/datasets/2b79923846fb52e62d686f2db846392575c8eb5b732d9d26cd3ca9378c622d40.87bd52d0f0ee055c1e455ef54ba45149d188552f07991b765da256a1b512ca0b.tfidf_vectors_sparse.npz
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/nmslib_index.bin not found in cache, downloading to /tmp/tmp8kfivzlr


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Models loaded


In [ ]:
# ── Cache ─────────────────────────────────────────────────────────────────────
umls_cache  = {}
embed_cache = {}

# ── Util functions ────────────────────────────────────────────────────────────
def extract_cuis(text):
    if text in umls_cache:
        return umls_cache[text]
    doc  = nlp(text)
    cuis = set()
    for ent in doc.ents:
        if ent._.kb_ents:
            for cui, score in ent._.kb_ents[:3]:
                if score > CUI_SCORE_THRESHOLD:
                    cuis.add(cui)
    umls_cache[text] = cuis
    return cuis


def get_embedding(text):
    if text in embed_cache:
        return embed_cache[text]
    emb = embed_model.encode([text], normalize_embeddings=True)[0]
    embed_cache[text] = emb
    return emb

print('✓ Util functions ready')

✓ Util functions ready


In [ ]:
# ── CUI-based chunking ────────────────────────────────────────────────────────
def build_umls_chunks(text):
    doc       = nlp(text)
    sentences = [s.text.strip() for s in doc.sents]

    chunks        = []
    current_chunk = []
    current_cuis  = set()

    for sent in sentences:
        sent_cuis = extract_cuis(sent)

        if current_chunk:
            chunk_text = ' '.join(current_chunk)
            overlap    = sent_cuis & current_cuis
            sim        = np.dot(get_embedding(chunk_text), get_embedding(sent))

            if (sim < SIM_THRESHOLD and len(overlap) == 0) or sim < 0.40:
                chunks.append(chunk_text)
                carry         = current_chunk[-OVERLAP_SENTENCES:]
                current_chunk = carry
                current_cuis  = extract_cuis(' '.join(carry)) if carry else set()

        current_chunk.append(sent)
        current_cuis.update(sent_cuis)

        if len(' '.join(current_chunk).split()) > CHUNK_WORD_LIMIT:
            chunks.append(' '.join(current_chunk))
            carry         = current_chunk[-OVERLAP_SENTENCES:]
            current_chunk = carry
            current_cuis  = extract_cuis(' '.join(carry)) if carry else set()

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

print('✓ CUI chunker ready')

✓ CUI chunker ready


In [ ]:
# ── Section splitting — title kept as part of body ────────────────────────────
def split_sections(text):
    pattern = r'---\s+([A-Za-z0-9 ]+)\s*(?:---)?'
    parts   = re.split(pattern, text)

    sections = []
    i        = 0

    while i < len(parts):
        part = parts[i].strip()

        # odd indices are captured title groups
        if i % 2 == 1:
            title    = part
            body     = parts[i + 1].strip() if i + 1 < len(parts) else ''
            combined = (title + '\n' + body).strip()
            if combined:
                sections.append(combined)
            i += 2
        else:
            # pre-header content with no title
            if part:
                sections.append(part)
            i += 1

    return sections

print('✓ Section splitter ready')

✓ Section splitter ready


In [ ]:
# ── Merge small chunks ────────────────────────────────────────────────────────
def merge_small_chunks(chunks, threshold=5, max_words=150):
    merged = []

    for c in chunks:
        wc = len(c.split())

        if wc <= threshold:
            if merged:
                combined = merged[-1] + ' ' + c
                if len(combined.split()) <= max_words:
                    merged[-1] = combined
                else:
                    doc           = nlp(merged[-1])
                    sents         = list(doc.sents)
                    last_sentence = sents[-1].text.strip() if sents else merged[-1]
                    merged.append((last_sentence + ' ' + c) if last_sentence else c)
            else:
                merged.append(c)
        else:
            merged.append(c)

    return merged

print('✓ Merge small chunks ready')

✓ Merge small chunks ready


In [ ]:
# ── Deduplication ─────────────────────────────────────────────────────────────
def deduplicate(chunks):
    seen   = set()
    unique = []
    for c in chunks:
        if c not in seen:
            unique.append(c)
            seen.add(c)
    return unique


# ── Main chunking logic ───────────────────────────────────────────────────────
def chunk_first_aid(text):
    sections     = split_sections(text)
    final_chunks = []

    for sec in tqdm(sections, desc='Chunking sections'):
        word_count = len(sec.split())

        # SMALL → keep as-is
        if word_count <= CHUNK_WORD_LIMIT:
            if word_count >= MIN_CHUNK_WORDS:
                final_chunks.append(sec)
            continue

        # LARGE → CUI chunking
        cui_chunks = build_umls_chunks(sec)
        cui_chunks = merge_small_chunks(cui_chunks)

        for c in cui_chunks:
            if len(c.split()) >= MIN_CHUNK_WORDS:
                final_chunks.append(c)

    return final_chunks

print('✓ Main chunking logic ready')

✓ Main chunking logic ready


In [ ]:
# ── Load text from Drive ──────────────────────────────────────────────────────
print(f'Reading {TXT_PATH}...')

with open(TXT_PATH, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f'✓ Loaded {len(raw_text):,} characters')

Reading /content/drive/MyDrive/merged_final3.txt...
✓ Loaded 4,494,520 characters


In [ ]:
# ── Chunking ──────────────────────────────────────────────────────────────────
print('Chunking with SECTION + CUI logic...\n')

chunks = chunk_first_aid(raw_text)
chunks = deduplicate(chunks)

print(f'\n✓ Total chunks after dedup: {len(chunks)}')

Chunking with SECTION + CUI logic...



Chunking sections:   0%|          | 0/7494 [00:00<?, ?it/s]


✓ Total chunks after dedup: 9228


In [ ]:
# ── Embeddings ────────────────────────────────────────────────────────────────
print('Generating embeddings...\n')

embeddings = embed_model.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
).astype('float32')

print(f'\n✓ Embeddings shape: {embeddings.shape}')

Generating embeddings...



Batches:   0%|          | 0/289 [00:00<?, ?it/s]


✓ Embeddings shape: (9228, 768)


In [ ]:
# ── FAISS ─────────────────────────────────────────────────────────────────────
print('Building FAISS index...')

dim        = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings)

print(f'✓ FAISS index: {faiss_index.ntotal} vectors  dim={dim}')

Building FAISS index...
✓ FAISS index: 9228 vectors  dim=768


In [ ]:
# ── BM25 ──────────────────────────────────────────────────────────────────────
print('Building BM25...')

def tokenize(text):
    return re.findall(r'\w+', text.lower())

tokenized = [tokenize(c) for c in chunks]
bm25      = BM25Okapi(tokenized)

print('✓ BM25 ready')

Building BM25...
✓ BM25 ready


In [ ]:
# ── Build metadata ────────────────────────────────────────────────────────────
metadata = []

for i, c in enumerate(chunks):
    metadata.append({
        'chunk_id': f'fa_{i}',
        'text'    : c,
        'length'  : len(c.split()),
        'source'  : 'first_aid',
        'title'   : '',
        'url'     : '',
        'section' : '',
    })

print(f'✓ Metadata built: {len(metadata)} entries')

✓ Metadata built: 9228 entries


In [ ]:
# ── Save all artifacts to Drive ───────────────────────────────────────────────
print('Saving to Drive...\n')

faiss.write_index(faiss_index, f'{DATA_DIR}/faiss.index')
print('✓ faiss.index saved')

with open(f'{DATA_DIR}/bm25.pkl', 'wb') as f:
    pickle.dump(bm25, f)
print('✓ bm25.pkl saved')

with open(f'{DATA_DIR}/metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print('✓ metadata.pkl saved')

with open(f'{DATA_DIR}/chunks.json', 'w') as f:
    json.dump(chunks, f, indent=2)
print('✓ chunks.json saved')

print(f'\n✅ DONE — DB saved to {DATA_DIR}')
print(f'   Total chunks : {len(chunks)}')
print(f'   Embedding dim: {dim}')
print(f'   Artifacts    : faiss.index, bm25.pkl, metadata.pkl, chunks.json')

Saving to Drive...

✓ faiss.index saved
✓ bm25.pkl saved
✓ metadata.pkl saved
✓ chunks.json saved

✅ DONE — DB saved to /content/drive/MyDrive/first_aid_db_4
   Total chunks : 9228
   Embedding dim: 768
   Artifacts    : faiss.index, bm25.pkl, metadata.pkl, chunks.json
